In [0]:
from pyspark.sql import functions as F
from functools import reduce
from operator import or_

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

DATASETS = ["flights", "weather", "airports"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:
total_count = 0

for dataset in DATASETS:
    df = spark.table(f"{BRONZE}.{dataset}")

    count = df.count() 
    print(f"\n{'=' * 60}")
    print(f"DATASET: {dataset.upper()}")
    print(f"{'=' * 60}")
    print(f"Count: {count}")
    total_count += count
    
    df.printSchema()
    


print(f"\n{'=' * 60}")
print(f"TOTAL COUNT: {total_count}")
print(f"{'=' * 60}:")


In [0]:
def _normalize_column_names(df):
    for column in df.columns:
        normalized = column.strip().lower().replace(" ", "_")

        if normalized != column:
            df = df.withColumnRenamed(column, normalized)

    return df


def _remove_exact_duplicates(df):
    metadata_columns = ["_ingested_at", "_source_dataset"]

    data_columns = [
        c for c in df.columns
        if c not in metadata_columns
    ]

    return df.dropDuplicates(data_columns) if data_columns else df


def _drop_missing_required(df, required_columns):
    if not required_columns:
        return df

    missing_columns = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns not found: {missing_columns}"
        )

    return df.dropna(how="any", subset=required_columns)


def _analyze_nulls(df, drop_empty_columns=True):
    counts = df.agg(
        F.count("*").alias("_row_count"),
        *[
            F.count(F.col(f"`{c.replace('`', '``')}`")).alias(f"_col_{i}")
            for i, c in enumerate(df.columns)
        ]
    ).first()

    row_count = counts["_row_count"]

    null_counts = {
        column: row_count - counts[f"_col_{i}"]
        for i, column in enumerate(df.columns)
    }

    total_nulls = sum(null_counts.values())

    empty_columns = [
        column
        for column, count in null_counts.items()
        if count == row_count
    ] if row_count > 0 else []

    print(f"Total null values: {total_nulls:,}")

    for column, count in null_counts.items():
        if count > 0:
            print(f"{column}: {count:,} nulls")

    if empty_columns:
        print(f"Entirely null columns: {empty_columns}")

    if drop_empty_columns:
        df = df.drop(*empty_columns)

    metrics = {
        "total_nulls": total_nulls,
        "null_counts": null_counts,
        "dropped_columns": empty_columns if drop_empty_columns else []
    }

    return df, metrics


def basic_checks(df, dataset, required_columns=None):
    df = _normalize_column_names(df)

    original_count = df.count()

    df = _remove_exact_duplicates(df)

    deduplicated_count = df.count()
    removed_count = original_count - deduplicated_count

    df = _drop_missing_required(df, required_columns)

    df, null_metrics = _analyze_nulls(
        df,
        drop_empty_columns=True
    )

    cleaned_count = deduplicated_count

    if required_columns:
        cleaned_count = df.count()

    missing_required_removed = deduplicated_count - cleaned_count
    null_count = null_metrics["total_nulls"]

    metrics = {
        "dataset": dataset,
        "original_rows": original_count,
        "cleaned_rows": cleaned_count,
        "duplicates_removed": removed_count,
        "missing_required_removed": missing_required_removed,
        "nulls": null_count,
        "null_counts": null_metrics["null_counts"],
        "dropped_columns": null_metrics["dropped_columns"]
    }

    print(
        f"[{dataset.upper()}] "
        f"Original: {original_count:,} | "
        f"Cleaned: {cleaned_count:,} | "
        f"Duplicates removed: {removed_count:,} | "
        f"Missing required removed: {missing_required_removed:,} | "
        f"Nulls: {null_count:,}"
    )

    return df, metrics


In [0]:
def upload_to_silver(df, dataset):
    target = f"{SILVER}.{dataset}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

In [0]:
def clean_airports(df):

    REQUIRED_AIRPORT_COLUMNS = [
    "airport_code",
    "latitude",
    "longitude"
    ]

    df, airports_metrics = basic_checks(
    airports,
    dataset="airports",
    required_columns=REQUIRED_AIRPORT_COLUMNS
    )

    df = df.withColumn(
        "latitude",
        F.when(
            F.col("latitude").between(-90, 90),
            F.col("latitude")
        )
    ).withColumn(
        "longitude",
        F.when(
            F.col("longitude").between(-180, 180),
            F.col("longitude")
        )
    ).withColumn(
        "elevation_ft",
        F.when(
            ~F.isnan("elevation_ft") &
            (F.abs(F.col("elevation_ft")) != float("inf")),
            F.col("elevation_ft")
        )
    )

    return df

In [0]:
airports = spark.table(f"{BRONZE}.airports")
airports_clean = clean_airports(airports)
upload_to_silver(airports_clean, "airports")

In [0]:
REQUIRED_FLIGHT_COLUMNS = [
    "flightdate",
    "reporting_airline",
    "flight_number_reporting_airline",
    "origin",
    "dest",
    "crsdeptime",
    "crsarrtime"
]


def _clean_flight_strings(df):
    # Trim whitespace and convert empty strings to null
    for name, dtype in df.dtypes:
        if dtype == "string":
            value = F.trim(F.col(name))

            df = df.withColumn(
                name,
                F.when(value != "", value)
            )

    # Standardize airport, airline, and aircraft identifiers
    codes = [
        "reporting_airline",
        "iata_code_reporting_airline",
        "origin",
        "dest",
        "tail_number",
        "cancellationcode"
    ]

    codes += [f"div{i}airport" for i in range(1, 6)]

    for name in codes:
        if name in df.columns:
            df = df.withColumn(
                name,
                F.upper(F.col(name))
            )

    return df


def _clean_flight_numbers(df):
    # Replace NaN and infinity with null
    numeric = [
        (name, dtype)
        for name, dtype in df.dtypes
        if dtype in ("double", "float")
    ]

    invalid_any = F.lit(False)

    for name, dtype in numeric:
        value = F.col(name)

        invalid = F.coalesce(
            F.isnan(value)
            | (F.abs(value) == float("inf")),
            F.lit(False)
        )

        invalid_any = invalid_any | invalid

    df = df.withColumn(
        "dq_nonfinite_numeric",
        invalid_any
    )

    for name, dtype in numeric:
        value = F.col(name)

        invalid = (
            F.isnan(value)
            | (F.abs(value) == float("inf"))
        )

        df = df.withColumn(
            name,
            F.when(~invalid, value)
             .otherwise(F.lit(None))
             .cast(dtype)
        )

    return df


def _valid_hhmm(name):
    clock = F.col(name)

    return (
        clock.isNotNull()
        & (clock == F.floor(clock))
        & (
            (
                clock.between(0, 2359)
                & ((clock % 100) < 60)
            )
            | (clock == 2400)
        )
    )


def _add_flight_schedule(df):
    # Parse source flight date
    df = df.withColumn(
        "flight_date",
        F.expr("try_cast(flightdate AS DATE)")
    )

    # Validate scheduled clock values
    df = (
        df.withColumn(
            "crsdeptime_valid",
            _valid_hhmm("crsdeptime")
        )
        .withColumn(
            "crsarrtime_valid",
            _valid_hhmm("crsarrtime")
        )
    )

    # Extract scheduled hours and minutes
    for source, prefix in [
        ("crsdeptime", "scheduled_departure"),
        ("crsarrtime", "scheduled_arrival")
    ]:
        clock = F.col(source)
        valid = F.col(f"{source}_valid")

        hour = F.when(
            clock == 2400,
            F.lit(0)
        ).otherwise(
            F.floor(clock / 100)
        ).cast("int")

        minute = (clock % 100).cast("int")

        df = (
            df.withColumn(
                f"{prefix}_hour",
                F.when(valid, hour)
            )
            .withColumn(
                f"{prefix}_minute",
                F.when(valid, minute)
            )
        )

    # 2400 represents midnight at the end of the date
    df = df.withColumn(
        "scheduled_departure_local_date",
        F.when(
            F.col("crsdeptime") == 2400,
            F.date_add("flight_date", 1)
        ).otherwise(F.col("flight_date"))
    )

    # Local timestamp, without timezone conversion
    df = df.withColumn(
        "scheduled_departure_local_ts",
        F.when(
            F.col("crsdeptime_valid")
            & F.col("flight_date").isNotNull(),
            F.to_timestamp_ntz(
                F.concat_ws(
                    " ",
                    F.date_format(
                        "scheduled_departure_local_date",
                        "yyyy-MM-dd"
                    ),
                    F.format_string(
                        "%02d:%02d:00",
                        F.col("scheduled_departure_hour"),
                        F.col("scheduled_departure_minute")
                    )
                ),
                F.lit("yyyy-MM-dd HH:mm:ss")
            )
        )
    )

    # Derive calendar fields for consistency checks
    df = (
        df.withColumn(
            "calendar_year",
            F.year("flight_date")
        )
        .withColumn(
            "calendar_quarter",
            F.quarter("flight_date")
        )
        .withColumn(
            "calendar_month",
            F.month("flight_date")
        )
        .withColumn(
            "calendar_dayofmonth",
            F.dayofmonth("flight_date")
        )
        .withColumn(
            "calendar_dayofweek",
            F.pmod(
                F.dayofweek("flight_date") + 5, 7
            ) + 1
        )
    )

    return df


def _add_flight_quality_flags(df):
    # Cancellation and diversion status
    df = (
        df.withColumn(
            "is_cancelled",
            F.when(
                F.col("cancelled").isin(0, 1),
                F.col("cancelled") == 1
            )
        )
        .withColumn(
            "is_diverted",
            F.when(
                F.col("diverted").isin(0, 1),
                F.col("diverted") == 1
            )
        )
    )

    df = df.withColumn(
        "flight_status",
        F.when(
            F.col("is_cancelled") & F.col("is_diverted"),
            "cancelled_and_diverted"
        )
        .when(F.col("is_cancelled"), "cancelled")
        .when(F.col("is_diverted"), "diverted")
        .when(
            (F.col("is_cancelled") == False)
            & (F.col("is_diverted") == False),
            "completed"
        )
        .otherwise("unknown")
    )

    # Invalid cancellation/diversion flags
    df = df.withColumn(
        "dq_invalid_status",
        ~F.coalesce(
            F.col("cancelled").isin(0, 1),
            F.lit(False)
        )
        | ~F.coalesce(
            F.col("diverted").isin(0, 1),
            F.lit(False)
        )
        | F.coalesce(
            F.col("is_cancelled") & F.col("is_diverted"),
            F.lit(False)
        )
    )

    # Compare reported calendar fields to flight date
    calendar_mismatch = F.lit(False)

    calendar_columns = {
        "year": "calendar_year",
        "source_year": "calendar_year",
        "quarter": "calendar_quarter",
        "month": "calendar_month",
        "source_month": "calendar_month",
        "dayofmonth": "calendar_dayofmonth",
        "dayofweek": "calendar_dayofweek"
    }

    for raw, derived in calendar_columns.items():
        if raw in df.columns:
            calendar_mismatch = calendar_mismatch | F.coalesce(
                F.col(raw).isNotNull()
                & (F.col(raw) != F.col(derived)),
                F.lit(False)
            )

    df = df.withColumn(
        "dq_calendar_mismatch",
        calendar_mismatch
    )

    # Validate nonnegative delay and duration fields
    nonnegative_columns = [
        "depdelayminutes",
        "arrdelayminutes",
        "carrierdelay",
        "weatherdelay",
        "nasdelay",
        "securitydelay",
        "lateaircraftdelay",
        "actualelapsedtime",
        "airtime",
        "taxiin",
        "taxiout"
    ]

    invalid_numeric = F.lit(False)

    for name in nonnegative_columns:
        invalid_numeric = invalid_numeric | F.coalesce(
            F.col(name) < 0,
            F.lit(False)
        )

    invalid_numeric = (
        invalid_numeric
        | F.coalesce(
            F.col("crselapsedtime") <= 0,
            F.lit(False)
        )
        | F.coalesce(
            F.col("distance") <= 0,
            F.lit(False)
        )
    )

    df = df.withColumn(
        "dq_invalid_numeric_value",
        invalid_numeric
    )

    # Validate actual operational clock times
    actual_clocks = [
        "deptime",
        "arrtime",
        "wheelsoff",
        "wheelson",
        "firstdeptime"
    ]

    for i in range(1, 6):
        actual_clocks.extend([
            f"div{i}wheelson",
            f"div{i}wheelsoff"
        ])

    invalid_clock = F.lit(False)

    for name in actual_clocks:
        if name in df.columns:
            invalid_clock = invalid_clock | F.coalesce(
                F.col(name).isNotNull()
                & ~_valid_hhmm(name),
                F.lit(False)
            )

    df = df.withColumn(
        "dq_invalid_actual_time",
        invalid_clock
    )

    # Check reported delay values against delay classifications
    inconsistent = F.lit(False)

    delay_columns = [
        ("depdelay", "depdelayminutes", "depdel15"),
        ("arrdelay", "arrdelayminutes", "arrdel15")
    ]

    for delay, minutes, flag in delay_columns:
        d = F.col(delay)
        m = F.col(minutes)
        f = F.col(flag)

        inconsistent = inconsistent | F.coalesce(
            (
                d.isNotNull()
                & m.isNotNull()
                & (
                    F.abs(
                        m - F.greatest(d, F.lit(0.0))
                    ) > 1
                )
            )
            | (
                d.isNotNull()
                & f.isNotNull()
                & (
                    f != F.when(d >= 15, 1.0)
                          .otherwise(0.0)
                )
            ),
            F.lit(False)
        )

        inconsistent = inconsistent | F.coalesce(
            f.isNotNull() & ~f.isin(0, 1),
            F.lit(False)
        )

    df = df.withColumn(
        "dq_delay_inconsistent",
        inconsistent
    )

    # Check gate-to-gate elapsed time consistency
    elapsed_columns = [
        "actualelapsedtime",
        "taxiout",
        "airtime",
        "taxiin"
    ]

    all_present = F.lit(True)

    for name in elapsed_columns:
        all_present = (
            all_present & F.col(name).isNotNull()
        )

    df = df.withColumn(
        "dq_elapsed_time_mismatch",
        F.coalesce(
            all_present
            & (F.col("is_cancelled") == False)
            & (F.col("is_diverted") == False)
            & (
                F.abs(
                    F.col("actualelapsedtime")
                    - F.col("taxiout")
                    - F.col("airtime")
                    - F.col("taxiin")
                ) > 2
            ),
            F.lit(False)
        )
    )

    # Validate cancellation reason codes
    code = F.col("cancellationcode")

    df = df.withColumn(
        "dq_cancellation_code",
        F.coalesce(
            (
                code.isNotNull()
                & ~code.isin("A", "B", "C", "D")
            )
            | (
                (F.col("is_cancelled") == True)
                & code.isNull()
            )
            | (
                (F.col("is_cancelled") == False)
                & code.isNotNull()
            ),
            F.lit(False)
        )
    )

    return df


def _add_flight_rejection_reasons(df):
    # Reject missing essential flight fields
    missing_required = F.lit(False)

    for name in REQUIRED_FLIGHT_COLUMNS:
        missing_required = (
            missing_required | F.col(name).isNull()
        )

    # Validate three-character airport identifiers
    invalid_origin = ~F.coalesce(
        F.col("origin").rlike("^[A-Z0-9]{3}$"),
        F.lit(False)
    )

    invalid_dest = ~F.coalesce(
        F.col("dest").rlike("^[A-Z0-9]{3}$"),
        F.lit(False)
    )

    # Build rejection reasons
    reasons = F.concat_ws(
        ";",
        F.when(
            missing_required,
            "missing_required_field"
        ),
        F.when(
            F.col("flight_date").isNull(),
            "invalid_flight_date"
        ),
        F.when(
            ~F.coalesce(
                F.col("crsdeptime_valid"),
                F.lit(False)
            ),
            "invalid_departure_time"
        ),
        F.when(
            ~F.coalesce(
                F.col("crsarrtime_valid"),
                F.lit(False)
            ),
            "invalid_arrival_time"
        ),
        F.when(
            invalid_origin | invalid_dest,
            "invalid_airport_code"
        ),
        F.when(
            F.col("flight_number_reporting_airline") <= 0,
            "invalid_flight_number"
        )
    )

    return df.withColumn(
        "_rejection_reason",
        F.when(F.length(reasons) > 0, reasons)
    )


def clean_flights(df):
    # 1. Normalize columns and source values
    df = _normalize_column_names(df)
    df = _clean_flight_strings(df)
    df = _clean_flight_numbers(df)

    # 2. Normalize schedule and calendar information
    df = _add_flight_schedule(df)

    # 3. Add data-quality flags
    df = _add_flight_quality_flags(df)

    # 4. Identify rejected records before filtering
    df = _add_flight_rejection_reasons(df)

    quality_flags = [
        "dq_nonfinite_numeric",
        "dq_invalid_status",
        "dq_calendar_mismatch",
        "dq_invalid_numeric_value",
        "dq_invalid_actual_time",
        "dq_delay_inconsistent",
        "dq_elapsed_time_mismatch",
        "dq_cancellation_code"
    ]

    # 5. Calculate source and quality metrics together
    summary = df.agg(
        F.count("*").alias("original_rows"),
        F.sum(
            F.col("_rejection_reason")
             .isNotNull()
             .cast("long")
        ).alias("rejected_rows"),
        *[
            F.sum(
                F.col(flag).cast("long")
            ).alias(flag)
            for flag in quality_flags
        ]
    ).first().asDict()

    original_count = summary["original_rows"]
    rejected_count = int(summary["rejected_rows"] or 0)

    # 6. Preserve rejected records
    rejected_df = df.filter(
        F.col("_rejection_reason").isNotNull()
    )

    # 7. Keep valid candidates for basic checks
    candidates = (
        df.filter(F.col("_rejection_reason").isNull())
          .drop("_rejection_reason")
    )

    # 8. Call the EXISTING basic_checks function
    clean_df, metrics = basic_checks(
        candidates,
        dataset="flights",
        required_columns=REQUIRED_FLIGHT_COLUMNS
    )

    # 9. Restore fully-null columns dropped by basic_checks
    # to retain a consistent flight schema
    for field in candidates.schema.fields:
        if field.name not in clean_df.columns:
            clean_df = clean_df.withColumn(
                field.name,
                F.lit(None).cast(field.dataType)
            )

    # 10. Update metrics with the complete source counts
    metrics["original_rows"] = original_count
    metrics["rejected_rows"] = rejected_count

    metrics["quality_flags"] = {
        flag: int(summary[flag] or 0)
        for flag in quality_flags
    }

    metrics["total_removed"] = (
        original_count - metrics["cleaned_rows"]
    )

    # 11. Print cleaning summary
    print(f"\n{'=' * 50}")
    print("FLIGHTS SILVER CLEANING")
    print(f"{'=' * 50}")

    print(f"Original: {metrics['original_rows']:,}")
    print(f"Cleaned: {metrics['cleaned_rows']:,}")
    print(f"Rejected: {metrics['rejected_rows']:,}")
    print(f"Duplicates removed: {metrics['duplicates_removed']:,}")
    print(f"Total removed: {metrics['total_removed']:,}")

    print("\nQuality flags:")

    for flag, count in metrics["quality_flags"].items():
        print(f"  {flag}: {count:,}")

    return clean_df, rejected_df, metrics


In [0]:
flights = spark.table(f"{BRONZE}.flights")
flights_clean, flights_rejected, flights_metrics = clean_flights(flights)

spark.sql(
    f"ALTER TABLE {SILVER}.flights SET TBLPROPERTIES "
    "('delta.feature.timestampNtz' = 'supported')"
)

upload_to_silver(flights_clean, "flights")

In [0]:
REQUIRED_WEATHER_COLUMNS = [
    "airport_code",
    "weather_timestamp"
]

WEATHER_MEASUREMENTS = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "visibility",
    "cloud_cover"
]


def _normalize_weather_codes(df):
    for name in ["airport_code", "airport"]:
        value = F.upper(F.trim(F.col(name)))

        df = df.withColumn(
            name,
            F.when(value != "", value)
        )

    df = df.withColumn(
        "dq_airport_mismatch",
        F.coalesce(
            F.col("airport_code").isNotNull()
            & F.col("airport").isNotNull()
            & (F.col("airport_code") != F.col("airport")),
            F.lit(False)
        )
    )

    return df.withColumn(
        "airport_code",
        F.coalesce(
            F.col("airport_code"),
            F.col("airport")
        )
    )


def _normalize_weather_time(df):
    df = df.withColumn(
        "weather_datetime",
        F.trim(F.col("weather_datetime"))
    )

    # Preserve recorded clock time without assuming UTC.
    df = df.withColumn(
        "weather_timestamp",
        F.expr(
            "try_cast(weather_datetime AS TIMESTAMP_NTZ)"
        )
    )

    df = df.withColumn(
        "dq_explicit_timezone",
        F.coalesce(
            F.col("weather_datetime").rlike(
                r"(?i)(?:z|[+-]\d{2}:?\d{2})$"
            ),
            F.lit(False)
        )
    )

    df = df.withColumn(
        "dq_non_hourly_time",
        F.coalesce(
            (F.minute("weather_timestamp") != 0)
            | (F.second("weather_timestamp") != 0)
            | (
                F.date_format(
                    "weather_timestamp", "SSSSSS"
                ) != "000000"
            ),
            F.lit(False)
        )
    )

    df = df.withColumn(
        "weather_year",
        F.year("weather_timestamp")
    )

    df = df.withColumn(
        "dq_year_mismatch",
        F.coalesce(
            F.col("year").isNotNull()
            & (F.col("year") != F.col("weather_year")),
            F.lit(False)
        )
    )

    return df


def _normalize_weather_measurements(df):
    numeric = [
        "latitude",
        "longitude",
        *WEATHER_MEASUREMENTS
    ]

    # Identify NaN/infinity before modifying measurements
    nonfinite = F.lit(False)

    for name in numeric:
        value = F.col(name)

        nonfinite = nonfinite | F.coalesce(
            F.isnan(value)
            | (F.abs(value) == float("inf")),
            F.lit(False)
        )

    df = df.withColumn(
        "dq_nonfinite_numeric",
        nonfinite
    )

    for name in numeric:
        value = F.col(name)

        invalid = F.coalesce(
            F.isnan(value)
            | (F.abs(value) == float("inf")),
            F.lit(False)
        )

        df = df.withColumn(
            name,
            F.when(
                invalid,
                F.lit(None).cast("double")
            ).otherwise(value)
        )

    # Validate geographic coordinates
    coordinate_invalid = F.lit(False)

    for name, low, high in [
        ("latitude", -90, 90),
        ("longitude", -180, 180)
    ]:
        value = F.col(name)

        coordinate_invalid = (
            coordinate_invalid | F.coalesce(
                value.isNotNull()
                & ~value.between(low, high),
                F.lit(False)
            )
        )

    df = df.withColumn(
        "dq_invalid_coordinates",
        coordinate_invalid
    )

    for name, low, high in [
        ("latitude", -90, 90),
        ("longitude", -180, 180)
    ]:
        df = df.withColumn(
            name,
            F.when(
                F.col(name).between(low, high),
                F.col(name)
            )
        )

    # Validate weather measurements
    nonnegative = [
        "precipitation",
        "rain",
        "snowfall",
        "wind_speed_10m",
        "wind_gusts_10m",
        "visibility"
    ]

    invalid_measurement = F.lit(False)

    for name in nonnegative:
        invalid_measurement = (
            invalid_measurement | F.coalesce(
                F.col(name) < 0,
                F.lit(False)
            )
        )

    invalid_measurement = invalid_measurement | F.coalesce(
        F.col("cloud_cover").isNotNull()
        & ~F.col("cloud_cover").between(0, 100),
        F.lit(False)
    )

    df = df.withColumn(
        "dq_invalid_measurement",
        invalid_measurement
    )

    for name in nonnegative:
        df = df.withColumn(
            name,
            F.when(
                F.col(name) >= 0,
                F.col(name)
            )
        )

    df = df.withColumn(
        "cloud_cover",
        F.when(
            F.col("cloud_cover").between(0, 100),
            F.col("cloud_cover")
        )
    )

    # Flag observations missing every weather measurement
    all_missing = F.lit(True)

    for name in WEATHER_MEASUREMENTS:
        all_missing = (
            all_missing & F.col(name).isNull()
        )

    return df.withColumn(
        "dq_all_measurements_missing",
        all_missing
    )


def _add_weather_rejection_reasons(df):
    airport_bad = ~F.coalesce(
        F.col("airport_code").rlike("^[A-Z0-9]{3}$"),
        F.lit(False)
    )

    reasons = F.concat_ws(
        ";",
        F.when(
            airport_bad,
            "missing_or_invalid_airport_code"
        ),
        F.when(
            F.col("weather_timestamp").isNull(),
            "invalid_weather_datetime"
        ),
        F.when(
            F.col("dq_explicit_timezone"),
            "timezone_offset_requires_review"
        ),
        F.when(
            F.col("dq_non_hourly_time"),
            "non_hourly_timestamp"
        )
    )

    return df.withColumn(
        "_rejection_reason",
        F.when(
            F.length(reasons) > 0,
            reasons
        )
    )


def clean_weather(df):
    # 1. Normalize columns and source values
    df = _normalize_column_names(df)
    df = _normalize_weather_codes(df)
    df = _normalize_weather_time(df)
    df = _normalize_weather_measurements(df)

    # 2. Identify invalid source records
    df = _add_weather_rejection_reasons(df)

    start = df.agg(
        F.count("*").alias("original_rows"),
        F.sum(
            F.col("_rejection_reason")
            .isNotNull()
            .cast("long")
        ).alias("invalid_rows")
    ).first()

    original_rows = int(start["original_rows"])
    invalid_rows = int(start["invalid_rows"] or 0)

    initially_rejected = df.filter(
        F.col("_rejection_reason").isNotNull()
    )

    candidates = (
        df.filter(F.col("_rejection_reason").isNull())
        .drop("_rejection_reason")
    )

    # 3. Detect duplicate airport-hour observations
    # Compare weather values, not partition metadata.
    payload = [
        "latitude",
        "longitude",
        *WEATHER_MEASUREMENTS
    ]

    candidates = candidates.withColumn(
        "_payload_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[F.col(c) for c in payload]
                ),
                {"ignoreNullFields": "false"}
            ),
            256
        )
    )

    key_columns = [
        "airport_code",
        "weather_timestamp"
    ]

    key_stats = (
        candidates.groupBy(*key_columns)
        .agg(
            F.count("*").alias("_rows_per_hour"),
            F.min("_payload_hash").alias("_min_hash"),
            F.max("_payload_hash").alias("_max_hash")
        )
        .withColumn(
            "_conflicting",
            F.col("_min_hash") != F.col("_max_hash")
        )
    )

    summary = key_stats.agg(
        F.count("*").alias("airport_hours"),
        F.sum(
            F.col("_conflicting").cast("long")
        ).alias("conflicting_hours"),
        F.sum(
            F.when(
                F.col("_conflicting"),
                F.col("_rows_per_hour")
            ).otherwise(0)
        ).alias("conflicting_rows"),
        F.sum(
            F.when(
                ~F.col("_conflicting"),
                F.col("_rows_per_hour") - 1
            ).otherwise(0)
        ).alias("duplicate_hour_rows_removed")
    ).first().asDict()

    joined = candidates.join(
        key_stats,
        key_columns,
        "inner"
    )

    # 4. Quarantine conflicting airport-hours
    conflicting = (
        joined.filter(F.col("_conflicting"))
        .withColumn(
            "_rejection_reason",
            F.lit("conflicting_airport_hour")
        )
        .select(*df.columns)
    )

    rejected = initially_rejected.unionByName(
        conflicting
    )

    # 5. Deduplicate matching airport-hours
    # Prefer records with matching directory metadata
    nonconflicting = (
        joined.filter(~F.col("_conflicting"))
        .withColumn(
            "_source_preference",
            F.when(
                F.col("airport") == F.col("airport_code"),
                2
            ).otherwise(0)
            + F.when(
                F.col("year") == F.col("weather_year"),
                1
            ).otherwise(0)
        )
    )

    unique_hours = (
        nonconflicting.selectExpr(
            "*",
            """
            ROW_NUMBER() OVER (
                PARTITION BY airport_code, weather_timestamp
                ORDER BY
                    _source_preference DESC,
                    _payload_hash ASC
            ) AS _row_number
            """
        )
        .filter(F.col("_row_number") == 1)
        .select(
            *[
                c for c in candidates.columns
                if c != "_payload_hash"
            ]
        )
    )

    # 6. Run existing general cleaning function
    clean, metrics = basic_checks(
        unique_hours,
        dataset="weather",
        required_columns=REQUIRED_WEATHER_COLUMNS
    )

    # Restore any entirely-null columns removed
    # to keep Silver schemas consistent.
    for field in unique_hours.schema.fields:
        if field.name not in clean.columns:
            clean = clean.withColumn(
                field.name,
                F.lit(None).cast(field.dataType)
            )

    cleaned_count = int(metrics["cleaned_rows"])

    if cleaned_count == 0:
        raise ValueError(
            "No valid weather records. Silver refresh aborted."
        )

    # 7. Calculate quality statistics
    quality_flags = [
        "dq_airport_mismatch",
        "dq_explicit_timezone",
        "dq_non_hourly_time",
        "dq_year_mismatch",
        "dq_nonfinite_numeric",
        "dq_invalid_coordinates",
        "dq_invalid_measurement",
        "dq_all_measurements_missing"
    ]

    flag_counts = clean.agg(
        *[
            F.sum(
                F.col(flag).cast("long")
            ).alias(flag)
            for flag in quality_flags
        ]
    ).first().asDict()

    # 8. Update audit metrics
    metrics.update({
        "original_rows": original_rows,
        "cleaned_rows": cleaned_count,
        "invalid_source_rows": invalid_rows,
        "conflicting_airport_hours": int(
            summary["conflicting_hours"] or 0
        ),
        "conflicting_source_rows": int(
            summary["conflicting_rows"] or 0
        ),
        "duplicate_airport_hours_removed": int(
            summary["duplicate_hour_rows_removed"] or 0
        ),
        "rejected_rows": (
            invalid_rows
            + int(summary["conflicting_rows"] or 0)
        ),
        "total_removed": original_rows - cleaned_count,
        "quality_flags": {
            flag: int(flag_counts[flag] or 0)
            for flag in quality_flags
        }
    })

    # 9. Print cleaning summary
    print(f"\n{'=' * 50}")
    print("WEATHER SILVER CLEANING")
    print(f"{'=' * 50}")

    print(f"Original: {metrics['original_rows']:,}")
    print(f"Cleaned: {metrics['cleaned_rows']:,}")
    print(f"Rejected: {metrics['rejected_rows']:,}")
    print(
        f"Duplicate airport-hours removed: "
        f"{metrics['duplicate_airport_hours_removed']:,}"
    )
    print(
        f"Conflicting airport-hours: "
        f"{metrics['conflicting_airport_hours']:,}"
    )
    print(f"Total removed: {metrics['total_removed']:,}")

    print("\nQuality flags:")

    for flag, count in metrics["quality_flags"].items():
        print(f"  {flag}: {count:,}")

    return clean, rejected, metrics



In [0]:
weather = spark.table(f"{BRONZE}.weather")
weather_clean, weather_rejected, weather_metrics = clean_weather(weather)
upload_to_silver(weather_clean, "weather")